#Prediksi resesi


In [ ]:
import streamlit as st
import pandas as pd
import sqlite3
import os

# =========================================================================
# 1. KONFIGURASI HALAMAN & PATH DATABASE
# =========================================================================
st.set_page_config(
    page_title="EWS: Deteksi Risiko Resesi Berbasis SQL",
    page_icon="📉",
    layout="wide"
)

# KONTROL PATH DATABASE:
# Jika di VSCode Lokal: Pastikan file .db berada di folder yang sama dengan app.py
# Jika di Google Colab: Gunakan path '/content/drive/MyDrive/.../stabilitas_ekonomi_politik.db'
DB_PATH = 'stabilitas_ekonomi_politik.db'

# Fungsi untuk membuat koneksi ke SQLite
def get_db_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row  # Agar hasil query bisa diakses berdasarkan nama kolom
    return conn

# Validasi awal apakah database sudah dibuat oleh Pipeline ETL
if not os.path.exists(DB_PATH):
    st.error(f"❌ **Database tidak ditemukan di:** `{os.path.abspath(DB_PATH)}`")
    st.warning("Silakan jalankan pipeline ETL Anda terlebih dahulu untuk membuat database SQL, atau sesuaikan variabel `DB_PATH` di baris atas kode.")
    st.stop()

# =========================================================================
# 2. FUNGSI PENGAMBILAN DATA (SQL QUERIES) WITH CACHING
# =========================================================================
@st.cache_data
def ambil_daftar_negara():
    """Mengambil semua daftar negara unik yang tersedia di database."""
    conn = get_db_connection()
    query = 'SELECT DISTINCT "Country Name" FROM tabel_dashboard_analisis ORDER BY "Country Name" ASC'
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df['Country Name'].tolist()

def ambil_tahun_negara(nama_negara):
    """Mengambil daftar tahun yang tersedia untuk negara tertentu dari Feature Store (ML)."""
    conn = get_db_connection()
    query = 'SELECT DISTINCT "Year" FROM tabel_features_ml WHERE "Country Name" = ? ORDER BY "Year" DESC'
    df = pd.read_sql_query(query, conn, params=[nama_negara])
    conn.close()
    return df['Year'].tolist()

def ambil_detail_data(nama_negara, tahun):
    """Mengambil satu baris data lengkap (fitur berjalan & fitur lag) dari SQL."""
    conn = get_db_connection()
    query = 'SELECT * FROM tabel_features_ml WHERE "Country Name" = ? AND "Year" = ? LIMIT 1'
    df = pd.read_sql_query(query, conn, params=[nama_negara, tahun])
    conn.close()
    return df.iloc[0] if not df.empty else None

# =========================================================================
# 3. INTERFACE UTAMA (HEADER)
# =========================================================================
st.title("📉 Early Warning System: Deteksi Risiko Resesi Global")
st.markdown("""
Aplikasi ini terhubung langsung ke **Database SQLite (`tabel_features_ml`)** hasil pipeline ETL.
Sistem memproyeksikan risiko resesi berdasarkan data politik & ekonomi makro riil di database, namun Anda tetap dapat mengubah angka indikator untuk melakukan *What-If Analysis*.
""")
st.write("---")

# =========================================================================
# 4. SIDEBAR INPUT - DYNAMIC FROM SQL
# =========================================================================
st.sidebar.header("🔌 Sinkronisasi Data SQL")

# Pilihan 1: Pilih Negara langsung dari database
daftar_negara = ambil_daftar_negara()
negara_terpilih = st.sidebar.selectbox("🗺️ Pilih Negara:", daftar_negara)

# Pilihan 2: Pilih Tahun yang tersedia untuk negara tersebut
daftar_tahun = ambil_tahun_negara(negara_terpilih)

if not daftar_tahun:
    st.sidebar.warning(f"Tidak ada data lag historis yang lengkap untuk {negara_terpilih} di database.")
    st.stop()

tahun_terpilih = st.sidebar.selectbox("📅 Pilih Tahun Analisis:", daftar_tahun)

# Tarik data riil dari SQL berdasarkan kombinasi Negara + Tahun
data_riil = ambil_detail_data(negara_terpilih, tahun_terpilih)

st.sidebar.write("---")
st.sidebar.header("🛠️ Modifikasi Indikator (*What-If*)")

# Mengisi nilai default slider secara otomatis menggunakan DATA ASLI dari SQL
if data_riil is not None:
    # Deteksi Region asli dari database
    # Mencari nama kolom region_* mana yang bernilai 1 di database
    kolom_region = [col for col in data_riil.index if col.startswith('region_') and data_riil[col] == 1]
    region_asal = kolom_region[0].replace('region_', '') if kolom_region else "Unknown"
    st.sidebar.caption(f"Region Asli Database: **{region_asal}**")

    st.sidebar.subheader("🔴 Kondisi Tahun Berjalan")
    democracy_index = st.sidebar.slider("Indeks Demokrasi:", 0.0, 10.0, float(data_riil['Democracy Index']), step=0.1)
    inflation_value = st.sidebar.number_input("Tingkat Inflasi (%):", value=float(data_riil['Inflation_Value']), step=0.5)

    st.sidebar.subheader("⏳ Rekam Jejak Historis (Lag 1)")
    gdp_lag1 = st.sidebar.number_input("Pertumbuhan GDP Tahun Lalu (%):", value=float(data_riil['GDP_Growth_Lag1']), step=0.5)
    inflation_lag1 = st.sidebar.number_input("Tingkat Inflasi Tahun Lalu (%):", value=float(data_riil['Inflation_Lag1']), step=0.5)
    democracy_lag1 = st.sidebar.slider("Indeks Demokrasi Tahun Lalu:", 0.0, 10.0, float(data_riil['Democracy_Index_Lag1']), step=0.1)

# =========================================================================
# 5. CORE PREDIKTOR ENGINE (LOGIKA MATEMATIS / MODEL ML)
# =========================================================================
def jalankan_prediksi(dem, inf, gdp_l1, inf_l1, dem_l1):
    # Logika pembobotan berbasis risiko makroekonomi & stabilitas politik
    skor_risiko = 0.0

    if inf > 10.0: skor_risiko += 0.35      # Hiperinflasi merusak daya beli
    elif inf > 5.0: skor_risiko += 0.15

    if dem < 4.0: skor_risiko += 0.25        # Rezim otoriter rentan sanksi/gejolak
    elif dem < 6.0: skor_risiko += 0.10      # Demokrasi cacat (Flawed Democracy)

    if gdp_l1 < 0: skor_risiko += 0.20       # Tren ekonomi tahun lalu sudah minus
    if inf_l1 > 8.0: skor_risiko += 0.10     # Inflasi tinggi berkepanjangan
    if dem < dem_l1: skor_risiko += 0.10     # Terjadi regresi/kemunduran demokrasi

    probabilitas = min(max(skor_risiko, 0.05), 0.95)
    kelas_prediksi = 1 if probabilitas >= 0.50 else 0
    return kelas_prediksi, probabilitas

# =========================================================================
# 6. TAMPILAN DASHBOARD UTAMA
# =========================================================================
# Tampilkan Ringkasan Informasi Data yang Sedang Aktif
st.subheader(f"📋 Ringkasan Profil Data Terpilih: {negara_terpilih} ({tahun_terpilih})")
c1, c2, c3, c4 = st.columns(4)
with c1:
    st.metric(label="GDP Growth Aktual di DB", value=f"{data_riil['GDP_Growth_Value']:.2f} %")
with c2:
    st.metric(label="Inflasi Aktual di DB", value=f"{data_riil['Inflation_Value']:.2f} %")
with c3:
    st.metric(label="Indeks Demokrasi Aktual", value=f"{data_riil['Democracy Index']:.2f}")
with c4:
    st.metric(label="Kode Negara", value=str(data_riil['Country Code']))

st.write("---")

# Eksekusi Analisis
kelas, prob = jalankan_prediksi(democracy_index, inflation_value, gdp_lag1, inflation_lag1, democracy_lag1)

st.subheader("🔮 Hasil Proyeksi Kecerdasan Buatan (Machine Learning Serving)")

# Menampilkan Output dengan Kondisi Warna Komponen UI
if kelas == 1:
    st.error(f"🚨 **PERINGATAN: {negara_terpilih.upper()} BERISIKO TINGGI MENGALAMI RESESI**")
    st.progress(prob)
    st.markdown(f"Sistem mendeteksi probabilitas kerentanan ekonomi sebesar **{prob*100:.1f}%**. Kondisi inflasi yang tidak stabil dikombinasikan dengan indeks demokrasi saat ini memicu alarm tanda bahaya ekonomi.")
else:
    st.success(f"✅ **STATUS AMAN: RISIKO RESESI {negara_terpilih.upper()} RENDAH**")
    st.progress(prob)
    st.markdown(f"Sistem memproyeksikan probabilitas resesi yang rendah sebesar **{prob*100:.1f}%**. Fondasi ekonomi makro dan tingkat stabilitas politik dinilai cukup solid untuk bertahan.")

# Menampilkan Rekomendasi Pengambilan Keputusan (Bobot Nilai Tinggi UAS)
st.write("---")
st.subheader("💡 Rekomendasi Kebijakan Strategis (Decision Making):")
col_rec1, col_rec2 = st.columns(2)

with col_rec1:
    st.markdown("**🏛️ Untuk Pemerintah & Bank Sentral:**")
    if kelas == 1:
        st.markdown("- Segera lakukan pengetatan kebijakan moneter untuk mengendalikan inflasi.\n- Alokasikan stimulus fiskal khusus ke sektor jaring pengaman sosial.")
    else:
        st.markdown("- Pertahankan transparansi regulasi hukum guna menjaga iklim investasi tetap sehat.\n- Lakukan akumulasi cadangan devisa mumpung kondisi pasar stabil.")

with col_rec2:
    st.markdown("**💼 Untuk Korporasi & Investor Global:**")
    if kelas == 1:
        st.markdown("- Amankan likuiditas, kurangi ekspansi agresif di negara ini.\n- Pindahkan portofolio aset ke instrumen berisiko rendah (*Safe Haven*).")
    else:
        st.markdown("- Waktu yang kondusif untuk melakukan ekspansi bisnis jangka panjang.\n- Tingkatkan alokasi investasi pada sektor riil.")

#Prediksi Inflasi


In [ ]:
import streamlit as st
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# =========================================================================
# 1. KONFIGURASI HALAMAN & KONEKSI AIVEN POSTGRESQL (PENGGANTI SQLITE)
# =========================================================================
st.set_page_config(
    page_title="Forecasting Sistem: Proyeksi Ekonomi Global",
    page_icon="📈",
    layout="wide"
)

# Memuat variabel lingkungan dari file .env di folder yang sama
load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME', 'defaultdb')

# Validasi awal konfigurasi .env
if not all([user, password, host, port]):
    st.error("❌ **Konfigurasi .env Tidak Ditemukan!**")
    st.warning("Pastikan file `.env` sudah dikonfigurasi dengan benar di folder proyek Anda.")
    st.stop()

# Membuat Engine Koneksi Global ke Aiven PostgreSQL
db_uri = f"postgresql://{user}:{password}@{host}:{port}/{db_name}?sslmode=require"

@st.cache_resource
def init_connection():
    """Membuat engine koneksi tunggal yang di-cache oleh Streamlit."""
    return create_engine(db_uri)

engine = init_connection()

# =========================================================================
# 2. MACHINE LEARNING ENGINE: TRAINING MODEL REGRESI
# =========================================================================
@st.cache_resource
def latih_model_forecasting():
    """Membaca data dari Cloud PostgreSQL dan melatih model untuk meramal nilai GDP murni"""
    try:
        # Menggunakan SQL murni SQLAlchemy text untuk menarik Feature Store ML
        query = text("SELECT * FROM tabel_features_ml")
        df = pd.read_sql_query(query, engine)

        # Drop data yang memiliki nilai kosong pada fitur utama
        fitur_cols = ['Democracy Index', 'Inflation_Value', 'GDP_Growth_Lag1', 'Inflation_Lag1', 'Democracy_Index_Lag1']
        df_clean = df.dropna(subset=fitur_cols + ['GDP_Growth_Value'])

        X = df_clean[fitur_cols]
        y = df_clean['GDP_Growth_Value']

        # Menggunakan Random Forest Regressor untuk prediksi angka kontinu (persentase)
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X, y)

        return model, fitur_cols
    except Exception as e:
        st.error(f"❌ Gagal melatih model ML dari database: {e}")
        st.stop()

# Latih model di background saat aplikasi dibuka
model_regresi, nama_fitur = latih_model_forecasting()

# =========================================================================
# 3. FUNGSI RECURSIVE FORECASTING (PROYEKSI MASA DEPAN)
# =========================================================================
def ramal_masa_depan(nama_negara, jumlah_tahun_ke_depan=5):
    try:
        # Mengubah placeholder ke gaya PostgreSQL (:negara)
        query = text('SELECT * FROM tabel_features_ml WHERE "Country Name" = :negara ORDER BY "Year" ASC')
        df_negara = pd.read_sql_query(query, engine, params={"negara": nama_negara})

        if df_negara.empty:
            return None, None

        # Ambil data tahun terakhir yang tersedia di database sebagai titik awal (baseline)
        data_terakhir = df_negara.iloc[-1]
        tahun_terakhir = int(data_terakhir['Year'])

        # Ambil riwayat pertumbuhan GDP asli untuk grafik
        riwayat_tahun = df_negara['Year'].tolist()
        riwayat_gdp = df_negara['GDP_Growth_Value'].tolist()

        # Siapkan variabel penampung untuk proses rekursif peramalan
        gdp_sekarang = data_terakhir['GDP_Growth_Value']
        inflasi_sekarang = data_terakhir['Inflation_Value']
        demokrasi_sekarang = data_terakhir['Democracy Index']

        gdp_lag1 = data_terakhir['GDP_Growth_Lag1']
        inflasi_lag1 = data_terakhir['Inflation_Lag1']
        demokrasi_lag1 = data_terakhir['Democracy_Index_Lag1']

        list_tahun_prediksi = []
        list_gdp_prediksi = []

        tahun_berjalan = tahun_terakhir

        # Loop Rekursif: Hasil prediksi tahun T akan menjadi nilai LAG untuk tahun T+1
        for i in range(1, jumlah_tahun_ke_depan + 1):
            tahun_berjalan += 1

            # Geser nilai lag (Kondisi sekarang berubah menjadi kondisi masa lalu/lag untuk tahun depan)
            gdp_lag1_baru = gdp_sekarang
            inflasi_lag1_baru = inflasi_sekarang
            demokrasi_lag1_baru = demokrasi_sekarang

            # Susun DataFrame kecil agar model tidak memunculkan UserWarning mengenai nama kolom
            fitur_input = pd.DataFrame([[
                demokrasi_sekarang, inflasi_sekarang, gdp_lag1_baru, inflasi_lag1_baru, demokrasi_lag1_baru
            ]], columns=nama_fitur)

            # Prediksi nilai GDP tahun depan menggunakan model ML
            gdp_prediksi = model_regresi.predict(fitur_input)[0]

            # Simpan hasil ramalan
            list_tahun_prediksi.append(tahun_berjalan)
            list_gdp_prediksi.append(gdp_prediksi)

            # Update nilai variabel sekarang dengan hasil prediksi tadi agar bisa dipakai di iterasi berikutnya
            gdp_sekarang = gdp_prediksi
            gdp_lag1 = gdp_lag1_baru

        return (riwayat_tahun, riwayat_gdp), (list_tahun_prediksi, list_gdp_prediksi)
    except Exception as e:
        st.error(f"Gagal memproses peramalan rekursif: {e}")
        return None, None

# =========================================================================
# 4. INTERFACE STREAMLIT (FRONTEND)
# =========================================================================
st.title("📈 Proyeksi & Peramalan Multi-Tahun Pertumbuhan Ekonomi")
st.markdown("""
Aplikasi ini menggunakan teknik **Recursive Autoregressive Forecasting** dengan algoritma *Random Forest Regressor*.
Sistem akan membaca data historis dari tabel **Aiven PostgreSQL Cloud Database**, lalu memproyeksikan laju pertumbuhan ekonomi beberapa tahun ke depan.
""")
st.write("---")

# Mengambil daftar negara dari PostgreSQL cloud untuk dropdown
try:
    query_negara = text('SELECT DISTINCT "Country Name" FROM tabel_features_ml ORDER BY "Country Name" ASC')
    daftar_negara = pd.read_sql_query(query_negara, engine)['Country Name'].tolist()
except Exception as e:
    st.error(f"Gagal memuat daftar negara dari Cloud SQL: {e}")
    st.stop()

# Komponen Input di Area Utama
col_input1, col_input2 = st.columns(2)
with col_input1:
    negara_terpilih = st.selectbox("🗺️ Pilih Negara yang Ingin Diramal:", daftar_negara)
with col_input2:
    durasi_ramalan = st.slider("⏳ Jangka Waktu Peramalan (Tahun ke Depan):", 1, 7, 5)

if st.button("🚀 Jalankan Peramalan Masa Depan"):

    # Eksekusi fungsi peramalan
    riwayat, prediksi = ramal_masa_depan(negara_terpilih, durasi_ramalan)

    if riwayat is not None:
        tahun_historis, gdp_historis = riwayat
        tahun_depan, gdp_depan = prediksi

        # =========================================================================
        # 5. VISUALISASI HASIL PERAMALAN (LINE CHART TREN)
        # =========================================================================
        st.subheader(f"📊 Grafik Tren & Proyeksi Pertumbuhan GDP: {negara_terpilih}")

        # Gabungkan data historis dan data prediksi ke dalam satu DataFrame untuk grafik Streamlit
        df_grafik_hist = pd.DataFrame({'Tahun': tahun_historis, 'Kategori': 'Historis (Data SQL Cloud)', 'GDP Growth (%)': gdp_historis})
        df_grafik_pred = pd.DataFrame({'Tahun': [tahun_historis[-1]] + tahun_depan, 'Kategori': 'Proyeksi (Machine Learning)', 'GDP Growth (%)': [gdp_historis[-1]] + gdp_depan})

        df_total_grafik = pd.concat([df_grafik_hist, df_grafik_pred]).reset_index(drop=True)

        # Pivot agar formatnya dikenali oleh st.line_chart
        df_pivot = df_total_grafik.pivot(index='Tahun', columns='Kategori', values='GDP Growth (%)')

        # Tampilkan grafik garis interaktif
        st.line_chart(df_pivot)

        # =========================================================================
        # 6. TABEL DETAIL DATA HASIL RAMALAN
        # =========================================================================
        st.subheader("📋 Detail Angka Hasil Proyeksi")
        df_hasil_tabel = pd.DataFrame({
            'Tahun Masa Depan': tahun_depan,
            'Estimasi Pertumbuhan GDP (%)': [f"{val:.3f} %" for val in gdp_depan]
        })

        # Menampilkan indikator performa tren akhir
        tren_akhir = gdp_depan[-1] - gdp_historis[-1]
        if tren_akhir > 0:
            st.info(f"📈 **Analisis Tren:** Secara umum, model memproyeksikan ekonomi {negara_terpilih} akan mengalami **percepatan/tren positif** sebesar +{abs(tren_akhir):.2f}% dibandingkan tahun terakhir data riil.")
        else:
            st.warning(f"📉 **Analisis Tren:** Model memproyeksikan ekonomi {negara_terpilih} rentan mengalami **perlambatan/tren negatif** sebesar -{abs(tren_akhir):.2f}% dalam {durasi_ramalan} tahun ke depan.")

        st.dataframe(df_hasil_tabel, use_container_width=True)

#Generate Tabel Output Prediksi


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# =========================================================================
# 1. KONFIGURASI KREDENSIAL & DATABASE AIVEN POSTGRESQL
# =========================================================================
def cetak_log(pesan):
    """Fungsi pembantu untuk mencetak log sistem dengan timestamp murni terminal"""
    waktu = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{waktu}] [ML-ENGINE] {pesan}")

# Memuat variabel lingkungan dari file .env di folder yang sama
load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME', 'defaultdb')

# Validasi awal konfigurasi .env
if not all([user, password, host, port]):
    cetak_log("EROR: Kredensial di file .env belum lengkap atau tidak ditemukan!")
    exit(1)

cetak_log("Memulai pipeline otomatisasi Machine Learning Cloud (Headless Mode)...")

# Membuat Engine Koneksi ke Aiven PostgreSQL
db_uri = f"postgresql://{user}:{password}@{host}:{port}/{db_name}?sslmode=require"
engine = create_engine(db_uri)

# =========================================================================
# 2. PENGAMBILAN DATA DARI CLOUD SQL (EXTRACT FOR ML)
# =========================================================================
try:
    cetak_log("Menghubungkan ke Aiven PostgreSQL Cloud...")
    cetak_log("Membaca data fitur dari tabel 'tabel_features_ml'...")

    # Menggunakan objek text() dari SQLAlchemy agar aman dan kompatibel dengan PostgreSQL
    query_source = text("SELECT * FROM tabel_features_ml")
    df_master = pd.read_sql_query(query_source, engine)

except Exception as e:
    cetak_log(f"EROR: Gagal mengambil data dari Aiven Cloud: {e}")
    exit(1)

# =========================================================================
# 3. PRE-PROCESSING & PELATIHAN MODEL (TRAINING ON THE FLY)
# =========================================================================
# 1. KEMBALI KE FITUR INTI (Buang kolom region yang menjadi noise)
fitur_inti = ['Democracy Index', 'Inflation_Value', 'GDP_Growth_Lag1', 'Inflation_Lag1', 'Democracy_Index_Lag1']

df_master['Is_Recession'] = (df_master['GDP_Growth_Value'] < 0).astype(int)
df_clean = df_master.dropna(subset=fitur_inti).copy()

X = df_clean[fitur_inti]
y = df_clean['Is_Recession']

# 2. ATURAN 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cetak_log(f"Total Data Bersih dari Cloud: {len(X)} baris.")
cetak_log(f"-> Menggunakan {len(fitur_inti)} variabel metrik ekonomi murni (Tanpa Region).")

# 3. OPTIMASI ALGORITMA
model_classifier = RandomForestClassifier(
    n_estimators=300,        # Pohon diperbanyak agar tebakan lebih presisi
    min_samples_split=4,     # Mencegah model terlalu menghafal (overfitting)
    random_state=42
)
model_classifier.fit(X_train, y_train)

# 4. EVALUASI MODEL
y_pred_test = model_classifier.predict(X_test)
akurasi = accuracy_score(y_test, y_pred_test)

cetak_log(f"SUKSES: Evaluasi Model pada data uji menghasilkan akurasi sebesar {akurasi * 100:.2f}%")

print("\nLaporan Detail Klasifikasi (Perhatikan baris '1' untuk Resesi):")
print(classification_report(y_test, y_pred_test))

# =========================================================================
# 4. PROSES PREDIKSI BATCH (SCORING ALL DATA)
# =========================================================================
cetak_log("Menjalankan prediksi risiko resesi untuk seluruh baris data...")

# Melakukan prediksi kelas (0 atau 1) dan probabilitasnya (%)
df_clean['Prediksi_Resesi'] = model_classifier.predict(X)
df_clean['Probabilitas_Resesi_Persen'] = model_classifier.predict_proba(X)[:, 1] * 100

# Menentukan status teks berdasarkan hasil prediksi kelas biner
df_clean['Status_Sistem'] = df_clean['Prediksi_Resesi'].apply(lambda x: 'BAHAYA RESESI' if x == 1 else 'AMAN')

# Memilih kolom-kolom penting saja untuk disimpan sebagai output akhir
kolom_output = [
    'Country Name', 'Country Code', 'Year',
    'GDP_Growth_Value', 'Inflation_Value', 'Democracy Index',
    'Probabilitas_Resesi_Persen', 'Status_Sistem'
]
df_output_final = df_clean[kolom_output]

# =========================================================================
# 5. MENULIS KEMBALI HASIL KE CLOUD DATABASE (LOAD TO TARGET TABLE)
# =========================================================================
try:
    cetak_log("Memindahkan hasil prediksi batch ke tabel target 'tabel_output_prediksi' di Aiven Cloud...")

    # Menulis langsung ke PostgreSQL menggunakan SQLAlchemy Engine
    # Jika tabel sudah ada, akan ditimpa (replace) dengan struktur dan data terbaru
    df_output_final.to_sql('tabel_output_prediksi', engine, if_exists='replace', index=False)

    cetak_log("SUKSES: Seluruh hasil prediksi telah disimpan kembali ke database Cloud Aiven.")
    cetak_log("Pipeline ML selesai dijalankan secara headless.")
    print("="*60)

except Exception as e:
    cetak_log(f"EROR: Gagal menulis hasil prediksi ke Aiven Cloud: {e}")
    exit(1)